In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import sys
import traceback
import zipfile

DATASET = Path('/kaggle/input/datasets/evanpieser/qcompile-verifiable')
COMPETITION = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
PUBLIC = DATASET / 'public_candidate'
SAMPLE = COMPETITION / 'sample_submission'
WORK = Path('/kaggle/working')
RELEASE = WORK / 'submission_files'
REFERENCE_SCORE = 3355599588.3155365
PUBLIC_REPORTED_SCORE = 426973779.3811362
sys.path.insert(0, str(COMPETITION / 'kickmix'))
sys.path.insert(0, str(DATASET / 'starter_kit'))
from verifier.verify import verify

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def assemble(source):
    if RELEASE.exists():
        shutil.rmtree(RELEASE)
    target = RELEASE / 'dfthcsos'
    target.mkdir(parents=True)
    for path in source.iterdir():
        if path.is_file() and (path.name == 'submission.npz' or path.name.startswith('angle_k_')):
            shutil.copy2(path, target / path.name)
    for name in ('dqi.kmb', 'invsqrt.kmb'):
        shutil.copy2(SAMPLE / name, RELEASE / name)
    return target


In [ ]:
candidate_error = None
verification = None
selected = None
try:
    provenance = json.loads((PUBLIC / 'full_verification.json').read_text())
    expected_hashes = {
        name: digest for name, digest in provenance['members'].items()
        if name.startswith('dfthcsos/')
    }
    actual_hashes = {
        str(path.relative_to(PUBLIC)): sha256(path)
        for path in (PUBLIC / 'dfthcsos').iterdir() if path.is_file()
    }
    assert actual_hashes == expected_hashes, 'public candidate hash mismatch'
    release_dfthc = assemble(PUBLIC / 'dfthcsos')
    candidate = verify('FeMoCo76o113e', release_dfthc)
    assert candidate['status'] == 'PASS'
    assert candidate['verified_shots'] == 70144
    assert (candidate['brot'], candidate['batch_size'], candidate['num_circuits']) == (8, 5, 15)
    assert candidate['resources']['angles_qubits'] == 123
    assert abs(candidate['score'] - PUBLIC_REPORTED_SCORE) < 1e-3
    assert candidate['score'] < REFERENCE_SCORE
    verification = candidate
    selected = 'verified_public_j5_candidate'
except Exception as exc:
    candidate_error = f'{type(exc).__name__}: {exc}'
    traceback.print_exc()
    release_dfthc = assemble(SAMPLE / 'dfthcsos')
    verification = verify('FeMoCo76o113e', release_dfthc)
    assert verification['status'] == 'PASS'
    assert verification['score'] <= REFERENCE_SCORE
    selected = 'official_baseline_fallback'
print('Selected:', selected)
print(json.dumps(verification, indent=2, default=float))


In [ ]:
files = sorted(path for path in RELEASE.rglob('*') if path.is_file())
members = {str(path.relative_to(RELEASE)): sha256(path) for path in files}
expected = {'dqi.kmb', 'invsqrt.kmb', 'dfthcsos/submission.npz'} | {
        f'dfthcsos/angle_k_{k}.kmb' for k in range(verification['num_circuits'])
    }
assert set(members) == expected
archive = WORK / 'submission.zip'
temporary = WORK / 'submission.tmp.zip'
with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for path in files:
        zf.write(path, str(path.relative_to(RELEASE)))
with zipfile.ZipFile(temporary) as zf:
    assert zf.testzip() is None
    assert set(zf.namelist()) == expected
os.replace(temporary, archive)
report = {
    'status': 'PASS',
    'selected_source': selected,
    'source_url': 'https://www.kaggle.com/code/lightningv08/bezout-schur-5-word-qrom',
    'candidate_error': candidate_error,
    'reference_score': REFERENCE_SCORE,
    'score': verification['score'],
    'reference_over_cost': REFERENCE_SCORE / verification['score'],
    'improvement_over_previous_8308120667': 8308120667.854655 / verification['score'],
    'projected_overall_with_reference_other_tasks_not_official': (REFERENCE_SCORE / verification['score']) ** (1 / 3),
    'verification': verification,
    'archive_sha256': sha256(archive),
    'member_sha256': members,
}
(WORK / 'release_verification.json').write_text(json.dumps(report, indent=2, default=float) + '\n')
if (PUBLIC / 'NOTICE').is_file():
    shutil.copy2(PUBLIC / 'NOTICE', WORK / 'NOTICE_dfthcsos.txt')
print('Archive:', archive, archive.stat().st_size, 'bytes')
print('dfthcsos score:', verification['score'])
print('Reference / cost:', report['reference_over_cost'])
print('Projected overall with reference other tasks (not official):', report['projected_overall_with_reference_other_tasks_not_official'])
